# FIFA World Cup 2026 — Tournament Prediction

End-to-end pipeline that:

1. **Trains** Random Forest models on historical World Cup match scores (1930–2022)
2. **Predicts** every 2026 group-stage fixture from `Data/clean_fifa_worldcup_fixture.csv`
3. **Builds** group standings (points + goal difference)
4. **Simulates** a simplified knockout bracket through to a predicted champion

**Outputs:** `Data/fifa_worldcup_2026_predictions.csv`, `Data/fifa_worldcup_2026_standings.csv`, `Data/predicted_tournament_results.csv`, `Data/final_tournament_standings.csv`

> **Note:** Qualification uses the top 16 teams overall (not FIFA's real group-winner + best-third rules). Knockout pairings follow standings order, not the official draw.

In [1]:
# Core libraries for data handling and score-prediction models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

## Step 1 — Train score-prediction models

Load historical matches and fit two separate regressors: one for home goals, one for away goals. Features are encoded team IDs only.

In [2]:
# Historical World Cup results used as training data
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna()

# Map team names to integer IDs; unknown teams get -1 (handled later in fixtures)
team_encoder = LabelEncoder()
df['HomeTeamEncoded'] = team_encoder.fit_transform(df['HomeTeam'])
df['AwayTeamEncoded'] = df['AwayTeam'].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)

X = df[['HomeTeamEncoded', 'AwayTeamEncoded']]
y_home = df['HomeGoals']
y_away = df['AwayGoals']

# 80/20 split for optional evaluation (models train on X_train below)
X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
X_train, X_test, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)

In [3]:
# Separate models let home/away scoring patterns differ
home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)

home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)


RandomForestRegressor(random_state=42)

## Step 2 — Predict 2026 group-stage fixtures

Apply the trained models to every scheduled 2026 match and save predicted scores.

In [4]:
fixtures = pd.read_csv("Data/clean_fifa_worldcup_fixture.csv")
fixtures.rename(columns={'home': 'HomeTeam', 'away': 'AwayTeam'}, inplace=True)

# Encode teams with the same encoder fitted on historical data
fixtures['HomeTeamEncoded'] = fixtures['HomeTeam'].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)
fixtures['AwayTeamEncoded'] = fixtures['AwayTeam'].apply(
    lambda x: team_encoder.transform([x])[0] if x in team_encoder.classes_ else -1
)

# Round continuous predictions to whole goals for standings math
fixtures['PredictedHomeGoals'] = home_model.predict(
    fixtures[['HomeTeamEncoded', 'AwayTeamEncoded']]
).round().astype(int)
fixtures['PredictedAwayGoals'] = away_model.predict(
    fixtures[['HomeTeamEncoded', 'AwayTeamEncoded']]
).round().astype(int)

fixtures['Winner'] = np.select(
    [
        fixtures['PredictedHomeGoals'] > fixtures['PredictedAwayGoals'],
        fixtures['PredictedHomeGoals'] < fixtures['PredictedAwayGoals'],
    ],
    [fixtures['HomeTeam'], fixtures['AwayTeam']],
    default='Draw',
)


In [5]:
# One row per fixture with predicted scores and winner
fixtures.to_csv("Data/fifa_worldcup_2026_predictions.csv", index=False)


## Step 3 — Compute group standings

Aggregate predicted results into points (3 win / 1 draw) and goal difference, then rank teams.

In [6]:
group_stage_results = pd.read_csv("Data/fifa_worldcup_2026_predictions.csv")
print(group_stage_results.head())

standings = {}  # team -> {Points, GD}

for _, row in group_stage_results.iterrows():
    home, away = row['HomeTeam'], row['AwayTeam']
    home_goals = int(row['PredictedHomeGoals'])
    away_goals = int(row['PredictedAwayGoals'])
    standings.setdefault(home, {'Points': 0, 'GD': 0})
    standings.setdefault(away, {'Points': 0, 'GD': 0})

    standings[home]['GD'] += home_goals - away_goals
    standings[away]['GD'] += away_goals - home_goals

    if home_goals > away_goals:
        standings[home]['Points'] += 3
    elif away_goals > home_goals:
        standings[away]['Points'] += 3
    else:
        standings[home]['Points'] += 1
        standings[away]['Points'] += 1

standings_df = pd.DataFrame.from_dict(standings, orient='index').reset_index().rename(columns={'index': 'Team'})
standings_df = standings_df.sort_values(by=['Points', 'GD'], ascending=False)


         HomeTeam     score        AwayTeam  year  HomeTeamEncoded  \
0          Mexico   Match 1    South Africa  2026               40   
1     South Korea   Match 2  Czech Republic  2026               65   
2  Czech Republic  Match 25    South Africa  2026               18   
3          Mexico  Match 28     South Korea  2026               40   
4  Czech Republic  Match 53          Mexico  2026               18   

   AwayTeamEncoded  PredictedHomeGoals  PredictedAwayGoals       Winner  
0               64                   1                   1         Draw  
1               18                   3                   1  South Korea  
2               64                   1                   1         Draw  
3               65                   2                   0       Mexico  
4               40                   1                   2       Mexico  


In [7]:
# Ranked table of all teams after the simulated group stage
standings_df.to_csv("Data/fifa_worldcup_2026_standings.csv", index=False)


## Step 4 — Simulate knockout stage

Take the top 16 teams from the standings table, retrain models (fresh encoder over all historical teams), then play out Round of 16 → Quarterfinals → Semifinals → Final.

Pairings follow standings order (1 vs 2, 3 vs 4, …). Ties in knockout matches are broken at random.

In [8]:
standings_df = pd.read_csv("Data/fifa_worldcup_2026_standings.csv")
qualified_teams = standings_df.head(16)['Team'].tolist()

print(f"\n🏆 Qualified Teams for Knockout Stage: {qualified_teams}")

# Retrain on full historical dataset with a unified team encoder
df = pd.read_csv("Data/clean_fifa_worldcup_matches.csv").dropna()

team_encoder = LabelEncoder()
all_teams = pd.concat([df['HomeTeam'], df['AwayTeam']]).unique()
team_encoder.fit(all_teams)

df['HomeTeamEncoded'] = team_encoder.transform(df['HomeTeam'])
df['AwayTeamEncoded'] = team_encoder.transform(df['AwayTeam'])

X = df[['HomeTeamEncoded', 'AwayTeamEncoded']]
y_home = df['HomeGoals']
y_away = df['AwayGoals']

X_train, X_test, y_train_home, y_test_home = train_test_split(X, y_home, test_size=0.2, random_state=42)
X_train, X_test, y_train_away, y_test_away = train_test_split(X, y_away, test_size=0.2, random_state=42)

home_model = RandomForestRegressor(n_estimators=100, random_state=42)
away_model = RandomForestRegressor(n_estimators=100, random_state=42)
home_model.fit(X_train, y_train_home)
away_model.fit(X_train, y_train_away)


def encode_team(team):
    """Return encoded team ID, or -1 if the team never appeared in historical data."""
    if team in team_encoder.classes_:
        return team_encoder.transform([team])[0]
    return -1


🏆 Qualified Teams for Knockout Stage: ['Brazil', 'Portugal', 'Mexico', 'Morocco', 'Sweden', 'Spain', 'Uruguay', 'Bosnia and Herzegovina', 'Ivory Coast', 'New Zealand', 'Argentina', 'England', 'Turkey', 'Austria', 'United States', 'Ecuador']


In [9]:
# Optional sanity check — training set size after the 80/20 split
print(X_train.shape, y_train_home.shape, y_train_away.shape)


(771, 2) (771,) (771,)


In [10]:
knockout_rounds = ["Round of 16", "Quarterfinals", "Semifinals", "Final"]
remaining_teams = qualified_teams.copy()

tournament_results = []  # every knockout match played
final_standings = []     # when each team was eliminated (or won)

for round_name in knockout_rounds:
    print(f"\n🔹 {round_name} Matches:")

    if len(remaining_teams) == 1:
        break

    next_round_teams = []

    # Odd team count: last team in the list gets a bye
    if len(remaining_teams) % 2 == 1:
        bye_team = remaining_teams.pop()
        print(f"⚠️ {bye_team} gets a bye and advances to the next round.")
        next_round_teams.append(bye_team)
        final_standings.append((bye_team, round_name))

    for i in range(0, len(remaining_teams), 2):
        team1, team2 = remaining_teams[i], remaining_teams[i + 1]
        encoded_team1, encoded_team2 = encode_team(team1), encode_team(team2)

        if encoded_team1 == -1 or encoded_team2 == -1:
            print(f"⚠️ Error: Team encoding issue with {team1} or {team2}")
            continue

        match_features = pd.DataFrame(
            [[encoded_team1, encoded_team2]],
            columns=['HomeTeamEncoded', 'AwayTeamEncoded'],
        )
        predicted_home_goals = home_model.predict(match_features)[0]
        predicted_away_goals = away_model.predict(match_features)[0]

        if predicted_home_goals > predicted_away_goals:
            winner, loser = team1, team2
        elif predicted_home_goals < predicted_away_goals:
            winner, loser = team2, team1
        else:
            winner = np.random.choice([team1, team2])  # no extra time / penalties modelled
            loser = team1 if winner == team2 else team2

        print(f"⚽ {team1} ({predicted_home_goals:.1f}) vs {team2} ({predicted_away_goals:.1f}) → Winner: {winner}")
        next_round_teams.append(winner)

        tournament_results.append([round_name, team1, team2, predicted_home_goals, predicted_away_goals, winner])
        final_standings.append((loser, round_name))

    remaining_teams = next_round_teams
    print(f"✅ Advancing Teams: {remaining_teams}")

final_winner = remaining_teams[0]
print(f"\n🏆🏆🏆 FIFA World Cup 2026 Champion: {final_winner} 🏆🏆🏆")
final_standings.append((final_winner, "Champion"))

results_df = pd.DataFrame(
    tournament_results,
    columns=["Round", "Team1", "Team2", "Predicted Goals Team1", "Predicted Goals Team2", "Winner"],
)
standings_df = pd.DataFrame(final_standings, columns=["Team", "Position"])

results_df.to_csv("Data/predicted_tournament_results.csv", index=False)
standings_df.to_csv("Data/final_tournament_standings.csv", index=False)

# Append knockout elimination data to the group-stage standings file
previous_standings = pd.read_csv("Data/fifa_worldcup_2026_standings.csv")
updated_standings = pd.concat([previous_standings, standings_df], ignore_index=True)
updated_standings.to_csv("Data/fifa_worldcup_2026_standings_Final_Updated.csv", index=False)



🔹 Round of 16 Matches:
⚽ Brazil (3.1) vs Portugal (1.7) → Winner: Brazil
⚽ Mexico (1.4) vs Morocco (1.4) → Winner: Morocco
⚽ Sweden (2.4) vs Spain (0.7) → Winner: Sweden
⚽ Uruguay (5.3) vs Bosnia and Herzegovina (0.3) → Winner: Uruguay
⚽ Ivory Coast (1.4) vs New Zealand (1.2) → Winner: Ivory Coast
⚽ Argentina (1.9) vs England (1.4) → Winner: Argentina
⚽ Turkey (1.7) vs Austria (0.8) → Winner: Turkey
⚽ United States (1.2) vs Ecuador (1.6) → Winner: Ecuador
✅ Advancing Teams: ['Brazil', 'Morocco', 'Sweden', 'Uruguay', 'Ivory Coast', 'Argentina', 'Turkey', 'Ecuador']

🔹 Quarterfinals Matches:
⚽ Brazil (1.5) vs Morocco (0.5) → Winner: Brazil
⚽ Sweden (2.3) vs Uruguay (0.2) → Winner: Sweden
⚽ Ivory Coast (1.1) vs Argentina (1.0) → Winner: Ivory Coast
⚽ Turkey (1.7) vs Ecuador (1.5) → Winner: Turkey
✅ Advancing Teams: ['Brazil', 'Sweden', 'Ivory Coast', 'Turkey']

🔹 Semifinals Matches:
⚽ Brazil (2.8) vs Sweden (1.5) → Winner: Brazil
⚽ Ivory Coast (2.9) vs Turkey (0.7) → Winner: Ivory Coast
